In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier

In [2]:
df_sales2020 = pd.read_csv('./추정매출 세분화/서울시_상권분석서비스(추정매출-상권)_2020년_filtered.csv', encoding='cp949')
df_sales2021 = pd.read_csv('./추정매출 세분화/서울시_상권분석서비스(추정매출-상권)_2021년_filtered.csv', encoding='cp949')
df_sales2022 = pd.read_csv('./추정매출 세분화/서울시_상권분석서비스(추정매출-상권)_2022년_filtered.csv', encoding='cp949')
df_sales2023 = pd.read_csv('./추정매출 세분화/서울시_상권분석서비스(추정매출-상권)_2023년_filtered.csv', encoding='cp949')
df_sales2024 = pd.read_csv('./추정매출 세분화/서울시_상권분석서비스(추정매출-상권)_2024년_filtered.csv', encoding='cp949')
df_store2020 = pd.read_csv('./점포-상권/서울시_상권분석서비스(점포-상권)_2020년_filtered.csv', encoding='cp949')
df_store2021 = pd.read_csv('./점포-상권/서울시_상권분석서비스(점포-상권)_2021년_filtered.csv', encoding='cp949')
df_store2022 = pd.read_csv('./점포-상권/서울시_상권분석서비스(점포-상권)_2022년_filtered.csv', encoding='cp949')
df_store2023 = pd.read_csv('./점포-상권/서울시_상권분석서비스(점포-상권)_2023년_filtered.csv', encoding='cp949')
df_store2024 = pd.read_csv('./점포-상권/서울시_상권분석서비스(점포-상권)_2024년_filtered.csv', encoding='cp949')

In [42]:
df_sales2025 = pd.read_csv('./서울시 상권분석서비스(추정매출-상권)_filtered.csv', encoding='cp949')

In [45]:
df_store2025 = pd.read_csv('./서울시 상권분석서비스(점포-상권)_filtered.csv', encoding='cp949')

In [43]:
df_sales2025 = df_sales2025[df_sales2025['기준_년분기_코드'].isin([20251, 20252])]

In [46]:
df_store2025 = df_store2025[df_store2025['기준_년분기_코드'].isin([20251,20252])]

In [47]:
store_2025 = build_yearly_store_from_quarter(df_store2025, 2025)
store_2025.head()

,상권_코드,store_avg,year
0,3001491,1684.5,2025
1,3001492,8475.0,2025
2,3001493,13189.5,2025
3,3001495,3050.5,2025
4,3001496,1038.5,2025


In [44]:
sales_2025 = build_yearly_sales_from_quarter(df_sales2025, 2025)
sales_2025.head()

,상권_코드,sales_month_avg,year
0,3001491,8.624212e+10,2025
1,3001492,4.674967e+11,2025
2,3001493,1.188573e+11,2025
3,3001495,3.622952e+11,2025
4,3001496,8.796124e+10,2025


In [3]:
def build_yearly_sales_from_quarter(df, year):
    # 1. 해당 연도 분기만
    df_y = df[df["기준_년분기_코드"] // 10 == year].copy()

    # 2. 상권 + 분기 + 업종별 매출 합산
    g = (df_y
         .groupby(["상권_코드", "기준_년분기_코드"], as_index=False)["당월_매출_금액"]
         .sum())

    # 3. 상권별로 분기 평균
    out = (g
           .groupby("상권_코드", as_index=False)["당월_매출_금액"]
           .mean()
           .rename(columns={"당월_매출_금액": "sales_month_avg"}))

    out["year"] = year
    return out


In [4]:
sales_2020 = build_yearly_sales_from_quarter(df_sales2020, 2020)
sales_2021 = build_yearly_sales_from_quarter(df_sales2021, 2021)
sales_2022 = build_yearly_sales_from_quarter(df_sales2022, 2022)
sales_2023 = build_yearly_sales_from_quarter(df_sales2023, 2023)
sales_2024 = build_yearly_sales_from_quarter(df_sales2024, 2024)

In [5]:
df_yearly_sales = pd.concat(
    [sales_2020, sales_2021, sales_2022, sales_2023, sales_2024],
    ignore_index=True
)
df_yearly_sales

,상권_코드,sales_month_avg,year
0,3001491,6.676159e+10,2020
1,3001492,2.604531e+11,2020
2,3001493,8.712480e+10,2020
3,3001495,1.905591e+11,2020
4,3001496,2.154922e+11,2020
...,...,...,...
7435,3130323,1.254762e+10,2024
7436,3130324,7.538584e+09,2024
7437,3130325,3.959156e+09,2024
7438,3130326,1.493443e+10,2024


In [6]:
df_yearly_sales = df_yearly_sales.sort_values(["상권_코드", "year"]).reset_index(drop=True)

# 전년 대비 변화율 (예: 2023이 2022보다 10% 증가 → 0.10)
df_yearly_sales["sales_yoy"] = (
    df_yearly_sales.groupby("상권_코드")["sales_month_avg"]
                   .pct_change()
)

df_yearly_sales[["상권_코드", "year", "sales_month_avg", "sales_yoy"]].head(15)


,상권_코드,year,sales_month_avg,sales_yoy
0,3001491,2020,6.676159e+10,NaN
1,3001491,2021,7.594446e+10,0.137547
2,3001491,2022,1.000172e+11,0.316978
3,3001491,2023,8.893227e+10,-0.110830
4,3001491,2024,9.216239e+10,0.036321
5,3001492,2020,2.604531e+11,NaN
6,3001492,2021,3.510482e+11,0.347836
7,3001492,2022,3.751974e+11,0.068792
8,3001492,2023,4.275657e+11,0.139575
9,3001492,2024,4.676735e+11,0.093805


In [7]:
df_yearly_sales["next_sales_yoy"] = (
    df_yearly_sales.groupby("상권_코드")["sales_yoy"]
                   .shift(-1)
)

df_yearly_sales[["상권_코드", "year", "sales_yoy", "next_sales_yoy"]].head(15)


,상권_코드,year,sales_yoy,next_sales_yoy
0,3001491,2020,NaN,0.137547
1,3001491,2021,0.137547,0.316978
2,3001491,2022,0.316978,-0.110830
3,3001491,2023,-0.110830,0.036321
4,3001491,2024,0.036321,NaN
5,3001492,2020,NaN,0.347836
6,3001492,2021,0.347836,0.068792
7,3001492,2022,0.068792,0.139575
8,3001492,2023,0.139575,0.093805
9,3001492,2024,0.093805,NaN


In [8]:
def label_sales_direction(next_sales_yoy, thr=0.03):
    if pd.isna(next_sales_yoy):
        return np.nan
    if next_sales_yoy >= thr:
        return "INC"   # 증가
    if next_sales_yoy <= -thr:
        return "DEC"   # 감소
    return "FLAT"      # 유지

df_yearly_sales["y_sales_dir"] = df_yearly_sales["next_sales_yoy"].apply(label_sales_direction)

df_yearly_sales["y_sales_dir"].value_counts(dropna=False)


y_sales_dir
INC     3350
DEC     1624
NaN     1488
FLAT     978
Name: count, dtype: int64

In [9]:
FEATURES = ["sales_month_avg", "sales_yoy"]

df_train = df_yearly_sales.dropna(subset=FEATURES + ["y_sales_dir"]).copy()

X = df_train[FEATURES]
y = df_train["y_sales_dir"]

print("X shape:", X.shape)
print("y distribution:\n", y.value_counts())


X shape: (4464, 2)
y distribution:
 y_sales_dir
INC     2174
DEC     1402
FLAT     888
Name: count, dtype: int64


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier

# X, y는 이미 만들어져 있다고 가정
# X = df_train[["sales_month_avg", "sales_yoy"]]
# y = df_train["y_sales_dir"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train, y_train)

pred = model.predict(X_test)
print(classification_report(y_test, pred))


              precision    recall  f1-score   support

         DEC       0.43      0.37      0.40       280
        FLAT       0.27      0.20      0.23       178
         INC       0.54      0.64      0.58       435

    accuracy                           0.47       893
   macro avg       0.41      0.41      0.41       893
weighted avg       0.45      0.47      0.46       893



In [13]:
import joblib
from pathlib import Path

MODELS_DIR = Path("./models")
MODELS_DIR.mkdir(exist_ok=True)

model_path = MODELS_DIR / "model_sales_direction.pkl"
joblib.dump(model, model_path)

print("Saved:", model_path.resolve())

Saved: C:\Users\user\Documents\GitHub\trade_area_analysis\data\models\model_sales_direction.pkl


In [14]:
loaded = joblib.load(model_path)

sample = X.head(5)
print("predict:", loaded.predict(sample))
# 확률도 보고 싶으면(선택)
print("proba:", loaded.predict_proba(sample))
print("classes:", loaded.classes_)


predict: ['INC' 'DEC' 'INC' 'INC' 'INC']
proba: [[0.08       0.11       0.81      ]
 [0.71333333 0.07333333 0.21333333]
 [0.07333333 0.11333333 0.81333333]
 [0.10666667 0.00333333 0.89      ]
 [0.05       0.15333333 0.79666667]]
classes: ['DEC' 'FLAT' 'INC']


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

# X, y는 이미 준비돼 있다고 가정
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ),
    "LogisticRegression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),
    "SVC_rbf": SVC(
        kernel="rbf",
        class_weight="balanced",
        probability=True,
        random_state=42
    ),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "DecisionTree": DecisionTreeClassifier(
        max_depth=None,
        class_weight="balanced",
        random_state=42
    )
}

for name, clf in models.items():
    print(f"\n===== {name} =====")
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    print(classification_report(y_test, pred))



===== RandomForest =====
              precision    recall  f1-score   support

         DEC       0.43      0.37      0.40       280
        FLAT       0.27      0.20      0.23       178
         INC       0.54      0.64      0.58       435

    accuracy                           0.47       893
   macro avg       0.41      0.41      0.41       893
weighted avg       0.45      0.47      0.46       893


===== LogisticRegression =====


C:\Users\user\miniconda3\envs\yh\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\user\miniconda3\envs\yh\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\user\miniconda3\envs\yh\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


              precision    recall  f1-score   support

         DEC       0.00      0.00      0.00       280
        FLAT       0.20      1.00      0.33       178
         INC       0.00      0.00      0.00       435

    accuracy                           0.20       893
   macro avg       0.07      0.33      0.11       893
weighted avg       0.04      0.20      0.07       893


===== SVC_rbf =====
              precision    recall  f1-score   support

         DEC       0.35      0.75      0.48       280
        FLAT       0.25      0.37      0.29       178
         INC       0.34      0.02      0.04       435

    accuracy                           0.32       893
   macro avg       0.31      0.38      0.27       893
weighted avg       0.33      0.32      0.23       893


===== KNN =====
              precision    recall  f1-score   support

         DEC       0.34      0.39      0.36       280
        FLAT       0.19      0.15      0.17       178
         INC       0.51      0.49    

In [19]:
def build_yearly_store_from_quarter(df, year):
    # 1) 해당 연도 분기만 선택
    df_y = df[df["기준_년분기_코드"] // 10 == year].copy()

    # 3) 상권별+분기별 점포 합 (업종행이 여러 개면 전부 합산)
    g = (df_y.groupby(["상권_코드", "기준_년분기_코드"], as_index=False)["점포_수"]
            .sum())

    # 4) 상권별로 분기 평균 → “그 해의 평균 점포 규모”
    out = (g.groupby("상권_코드", as_index=False)["점포_수"]
             .mean()
             .rename(columns={"점포_수": "store_avg"}))

    out["year"] = year
    return out


In [20]:
store_2020 = build_yearly_store_from_quarter(df_store2020, 2020)
store_2021 = build_yearly_store_from_quarter(df_store2021, 2021)
store_2022 = build_yearly_store_from_quarter(df_store2022, 2022)
store_2023 = build_yearly_store_from_quarter(df_store2023, 2023)
store_2024 = build_yearly_store_from_quarter(df_store2024, 2024)

df_yearly_store = pd.concat(
    [store_2020, store_2021, store_2022, store_2023, store_2024],
    ignore_index=True
)

df_yearly_store.head()


,상권_코드,store_avg,year
0,3001491,1578.00,2020
1,3001492,8740.50,2020
2,3001493,15009.25,2020
3,3001495,2917.75,2020
4,3001496,1061.50,2020


In [21]:
df_yearly_sales

,상권_코드,sales_month_avg,year,sales_yoy,next_sales_yoy,y_sales_dir
0,3001491,6.676159e+10,2020,NaN,0.137547,INC
1,3001491,7.594446e+10,2021,0.137547,0.316978,INC
2,3001491,1.000172e+11,2022,0.316978,-0.110830,DEC
3,3001491,8.893227e+10,2023,-0.110830,0.036321,INC
4,3001491,9.216239e+10,2024,0.036321,NaN,NaN
...,...,...,...,...,...,...
7435,3130327,1.119310e+10,2020,NaN,0.682849,INC
7436,3130327,1.883629e+10,2021,0.682849,-0.154464,DEC
7437,3130327,1.592677e+10,2022,-0.154464,-0.096033,DEC
7438,3130327,1.439728e+10,2023,-0.096033,-0.060688,DEC


In [22]:
df_year = pd.merge(
    df_yearly_sales,      # 매출(연 단위)
    df_yearly_store,      # 점포(연 단위)
    on=["상권_코드", "year"],
    how="inner"
)

df_year = df_year.sort_values(["상권_코드", "year"]).reset_index(drop=True)
df_year.head()

,상권_코드,sales_month_avg,year,sales_yoy,next_sales_yoy,y_sales_dir,store_avg
0,3001491,6.676159e+10,2020,NaN,0.137547,INC,1578.00
1,3001491,7.594446e+10,2021,0.137547,0.316978,INC,1550.25
2,3001491,1.000172e+11,2022,0.316978,-0.110830,DEC,1633.75
3,3001491,8.893227e+10,2023,-0.110830,0.036321,INC,1691.75
4,3001491,9.216239e+10,2024,0.036321,NaN,NaN,1703.00


In [23]:
# 점포 전년 대비 변화율
df_year["store_yoy"] = df_year.groupby("상권_코드")["store_avg"].pct_change()

# 다음 해 변화율 (라벨 만들려고)
df_year["next_store_yoy"] = df_year.groupby("상권_코드")["store_yoy"].shift(-1)
df_year["next_sales_yoy"] = df_year.groupby("상권_코드")["sales_yoy"].shift(-1)

df_year[["상권_코드","year","store_yoy","sales_yoy","next_store_yoy","next_sales_yoy"]].head(15)

,상권_코드,year,store_yoy,sales_yoy,next_store_yoy,next_sales_yoy
0,3001491,2020,NaN,NaN,-0.017586,0.137547
1,3001491,2021,-0.017586,0.137547,0.053862,0.316978
2,3001491,2022,0.053862,0.316978,0.035501,-0.110830
3,3001491,2023,0.035501,-0.110830,0.006650,0.036321
4,3001491,2024,0.006650,0.036321,NaN,NaN
5,3001492,2020,NaN,NaN,-0.023826,0.347836
6,3001492,2021,-0.023826,0.347836,0.002930,0.068792
7,3001492,2022,0.002930,0.068792,0.003214,0.139575
8,3001492,2023,0.003214,0.139575,-0.001747,0.093805
9,3001492,2024,-0.001747,0.093805,NaN,NaN


In [24]:
def label_saturation(next_store_yoy, next_sales_yoy):
    if pd.isna(next_store_yoy) or pd.isna(next_sales_yoy):
        return np.nan
    
    # 기준은 너무 예민하지 않게 2% 정도로
    if next_store_yoy > 0.02 and next_sales_yoy < -0.02:
        return "HIGH"
    if next_store_yoy > 0.02 and abs(next_sales_yoy) <= 0.02:
        return "MID"
    return "LOW"

df_year["y_saturation"] = df_year.apply(
    lambda r: label_saturation(r["next_store_yoy"], r["next_sales_yoy"]),
    axis=1
)

df_year["y_saturation"].value_counts(dropna=False)


y_saturation
LOW     5173
NaN     1488
HIGH     592
MID      187
Name: count, dtype: int64

In [25]:
FEATURES_SATU = ["sales_month_avg", "sales_yoy", "store_avg", "store_yoy"]

df_train2 = df_year.dropna(subset=FEATURES_SATU + ["y_saturation"]).copy()
X2 = df_train2[FEATURES_SATU]
y2 = df_train2["y_saturation"]

print("X2:", X2.shape)
print("y2:\n", y2.value_counts())


X2: (4464, 4)
y2:
 y_saturation
LOW     3798
HIGH     504
MID      162
Name: count, dtype: int64


In [26]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

model_satu = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model_satu.fit(X2_train, y2_train)

pred2 = model_satu.predict(X2_test)
print(classification_report(y2_test, pred2))


              precision    recall  f1-score   support

        HIGH       0.00      0.00      0.00       101
         LOW       0.85      1.00      0.92       760
         MID       0.00      0.00      0.00        32

    accuracy                           0.85       893
   macro avg       0.28      0.33      0.31       893
weighted avg       0.72      0.85      0.78       893



In [27]:
from pathlib import Path
import joblib

MODELS_DIR = Path("./models")
MODELS_DIR.mkdir(exist_ok=True)

path2 = MODELS_DIR / "model_saturation.pkl"
joblib.dump(model_satu, path2)

print("Saved:", path2.resolve())


Saved: C:\Users\user\Documents\GitHub\trade_area_analysis\data\models\model_saturation.pkl


In [28]:
# X2, y2 는 이미 준비돼 있다고 가정
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

models2 = {
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ),
    "SVC_rbf": SVC(
        kernel="rbf",
        class_weight="balanced",
        probability=True,
        random_state=42
    ),
    "KNN": KNeighborsClassifier(
        n_neighbors=5
    ),
    "DecisionTree": DecisionTreeClassifier(
        max_depth=None,
        class_weight="balanced",
        random_state=42
    )
}

for name, clf in models.items():
    print(f"\n===== {name} =====")
    clf.fit(X2_train, y2_train)
    pred = clf.predict(X2_test)
    print(classification_report(y2_test, pred))


===== RandomForest =====
              precision    recall  f1-score   support

        HIGH       0.00      0.00      0.00       101
         LOW       0.85      1.00      0.92       760
         MID       0.00      0.00      0.00        32

    accuracy                           0.85       893
   macro avg       0.28      0.33      0.31       893
weighted avg       0.72      0.85      0.78       893


===== LogisticRegression =====
              precision    recall  f1-score   support

        HIGH       0.00      0.00      0.00       101
         LOW       0.00      0.00      0.00       760
         MID       0.04      1.00      0.07        32

    accuracy                           0.04       893
   macro avg       0.01      0.33      0.02       893
weighted avg       0.00      0.04      0.00       893


===== SVC_rbf =====


C:\Users\user\miniconda3\envs\yh\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\user\miniconda3\envs\yh\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\user\miniconda3\envs\yh\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


              precision    recall  f1-score   support

        HIGH       0.13      0.81      0.22       101
         LOW       0.89      0.02      0.04       760
         MID       0.05      0.34      0.09        32

    accuracy                           0.12       893
   macro avg       0.35      0.39      0.12       893
weighted avg       0.77      0.12      0.06       893


===== KNN =====
              precision    recall  f1-score   support

        HIGH       0.12      0.03      0.05       101
         LOW       0.85      0.97      0.91       760
         MID       0.00      0.00      0.00        32

    accuracy                           0.83       893
   macro avg       0.32      0.33      0.32       893
weighted avg       0.74      0.83      0.78       893


===== DecisionTree =====
              precision    recall  f1-score   support

        HIGH       0.19      0.15      0.17       101
         LOW       0.85      0.88      0.86       760
         MID       0.06      0.0

C:\Users\user\miniconda3\envs\yh\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\user\miniconda3\envs\yh\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\user\miniconda3\envs\yh\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [29]:
df_change = pd.read_csv('./서울시 상권분석서비스(상권변화지표-상권)_filtered.csv', encoding='cp949')

In [32]:
df_change.columns

Index(['기준_년분기_코드', '상권_구분_코드', '상권_구분_코드_명', '상권_코드', '상권_코드_명', '상권_변화_지표',
       '상권_변화_지표_명', '운영_영업_개월_평균', '폐업_영업_개월_평균', '서울_운영_영업_개월_평균',
       '서울_폐업_영업_개월_평균'],
      dtype='object')

In [33]:
# 연도 컬럼 만들기 (20231 → 2023)
df_change["year"] = df_change["기준_년분기_코드"] // 10

# 상권 + 연도 기준으로 변화 상태 최빈값
df_change_year = (
    df_change
    .groupby(["상권_코드", "year"])["상권_변화_지표_명"]
    .agg(lambda x: x.value_counts().idxmax())
    .reset_index()
    .rename(columns={"상권_변화_지표_명": "y_change"})
)

df_change_year.head()


,상권_코드,year,y_change
0,3001491,2019,정체
1,3001491,2020,상권확장
2,3001491,2021,상권확장
3,3001491,2022,상권확장
4,3001491,2023,상권확장


In [34]:
df_change_year["y_change"].value_counts()


y_change
다이나믹    3366
정체      3236
상권축소    2321
상권확장    1493
Name: count, dtype: int64

In [35]:
df_change_model = pd.merge(
    df_year,
    df_change_year,
    on=["상권_코드", "year"],
    how="inner"
)

df_change_model.head()


,상권_코드,sales_month_avg,year,sales_yoy,next_sales_yoy,y_sales_dir,store_avg,store_yoy,next_store_yoy,y_saturation,y_change
0,3001491,6.676159e+10,2020,NaN,0.137547,INC,1578.00,NaN,-0.017586,LOW,상권확장
1,3001491,7.594446e+10,2021,0.137547,0.316978,INC,1550.25,-0.017586,0.053862,LOW,상권확장
2,3001491,1.000172e+11,2022,0.316978,-0.110830,DEC,1633.75,0.053862,0.035501,HIGH,상권확장
3,3001491,8.893227e+10,2023,-0.110830,0.036321,INC,1691.75,0.035501,0.006650,LOW,상권확장
4,3001491,9.216239e+10,2024,0.036321,NaN,NaN,1703.00,0.006650,NaN,NaN,상권확장


In [36]:
FEATURES_CHANGE = [
    "sales_month_avg",
    "sales_yoy",
    "store_avg",
    "store_yoy",
]

df_train3 = df_change_model.dropna(
    subset=FEATURES_CHANGE + ["y_change"]
).copy()

X3 = df_train3[FEATURES_CHANGE]
y3 = df_train3["y_change"]

print("X3 shape:", X3.shape)
print("y3 distribution:\n", y3.value_counts())


X3 shape: (5952, 4)
y3 distribution:
 y_change
정체      1904
다이나믹    1893
상권축소    1329
상권확장     826
Name: count, dtype: int64


In [37]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier

X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y3,
    test_size=0.2,
    random_state=42,
    stratify=y3
)

model_change = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model_change.fit(X3_train, y3_train)

pred3 = model_change.predict(X3_test)
print(classification_report(y3_test, pred3))


              precision    recall  f1-score   support

        다이나믹       0.48      0.55      0.51       379
        상권축소       0.35      0.29      0.32       266
        상권확장       0.33      0.16      0.21       165
          정체       0.49      0.58      0.53       381

    accuracy                           0.45      1191
   macro avg       0.41      0.40      0.39      1191
weighted avg       0.43      0.45      0.44      1191



In [40]:
from pathlib import Path
import joblib

MODELS_DIR = Path("./models")
MODELS_DIR.mkdir(exist_ok=True)

path3 = MODELS_DIR / "model_change_state.pkl"
joblib.dump(model_change, path3)

print("Saved:", path3.resolve())


Saved: C:\Users\user\Documents\GitHub\trade_area_analysis\data\models\model_change_state.pkl


In [39]:
models3 = {
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ),
    "SVC_rbf": SVC(
        kernel="rbf",
        class_weight="balanced",
        probability=True,
        random_state=42
    ),
    "KNN": KNeighborsClassifier(
        n_neighbors=5
    ),
    "DecisionTree": DecisionTreeClassifier(
        max_depth=None,
        class_weight="balanced",
        random_state=42
    )
}

for name, clf in models.items():
    print(f"\n===== {name} =====")
    clf.fit(X3_train, y3_train)
    pred33 = clf.predict(X3_test)
    print(classification_report(y3_test, pred33))


===== RandomForest =====
              precision    recall  f1-score   support

        다이나믹       0.48      0.55      0.51       379
        상권축소       0.35      0.29      0.32       266
        상권확장       0.33      0.16      0.21       165
          정체       0.49      0.58      0.53       381

    accuracy                           0.45      1191
   macro avg       0.41      0.40      0.39      1191
weighted avg       0.43      0.45      0.44      1191


===== LogisticRegression =====
              precision    recall  f1-score   support

        다이나믹       0.00      0.00      0.00       379
        상권축소       0.00      0.00      0.00       266
        상권확장       0.00      0.00      0.00       165
          정체       0.32      1.00      0.48       381

    accuracy                           0.32      1191
   macro avg       0.08      0.25      0.12      1191
weighted avg       0.10      0.32      0.16      1191


===== SVC_rbf =====


C:\Users\user\miniconda3\envs\yh\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\user\miniconda3\envs\yh\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\user\miniconda3\envs\yh\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


              precision    recall  f1-score   support

        다이나믹       0.41      0.14      0.21       379
        상권축소       0.25      0.89      0.39       266
        상권확장       0.19      0.05      0.08       165
          정체       0.42      0.06      0.10       381

    accuracy                           0.27      1191
   macro avg       0.32      0.29      0.20      1191
weighted avg       0.34      0.27      0.20      1191


===== KNN =====
              precision    recall  f1-score   support

        다이나믹       0.30      0.41      0.35       379
        상권축소       0.24      0.24      0.24       266
        상권확장       0.18      0.12      0.14       165
          정체       0.29      0.22      0.25       381

    accuracy                           0.27      1191
   macro avg       0.25      0.25      0.24      1191
weighted avg       0.27      0.27      0.26      1191


===== DecisionTree =====
              precision    recall  f1-score   support

        다이나믹       0.42      0.4

In [48]:
df_2025 = pd.merge(
    sales_2025,
    store_2025,
    on=["상권_코드", "year"],
    how="inner"
)

df_2025.head()

,상권_코드,sales_month_avg,year,store_avg
0,3001491,8.624212e+10,2025,1684.5
1,3001492,4.674967e+11,2025,8475.0
2,3001493,1.188573e+11,2025,13189.5
3,3001495,3.622952e+11,2025,3050.5
4,3001496,8.796124e+10,2025,1038.5


In [49]:
df_2024_2025 = pd.concat(
    [
        df_year[df_year["year"] == 2024][
            ["상권_코드", "year", "sales_month_avg", "store_avg"]
        ],
        df_2025[
            ["상권_코드", "year", "sales_month_avg", "store_avg"]
        ]
    ],
    ignore_index=True
)

df_2024_2025 = df_2024_2025.sort_values(
    ["상권_코드", "year"]
).reset_index(drop=True)


In [50]:
df_2024_2025["sales_yoy"] = (
    df_2024_2025.groupby("상권_코드")["sales_month_avg"]
                .pct_change()
)

df_2024_2025["store_yoy"] = (
    df_2024_2025.groupby("상권_코드")["store_avg"]
                .pct_change()
)

# 2025년 행만 추출
df_2025_X = df_2024_2025[df_2024_2025["year"] == 2025].copy()

df_2025_X.head()
df_2024_2025["sales_yoy"] = (
    df_2024_2025.groupby("상권_코드")["sales_month_avg"]
                .pct_change()
)

df_2024_2025["store_yoy"] = (
    df_2024_2025.groupby("상권_코드")["store_avg"]
                .pct_change()
)

# 2025년 행만 추출
df_2025_X = df_2024_2025[df_2024_2025["year"] == 2025].copy()

df_2025_X.head()


,상권_코드,year,sales_month_avg,store_avg,sales_yoy,store_yoy
1,3001491,2025,8.624212e+10,1684.5,-0.064237,-0.010863
3,3001492,2025,4.674967e+11,8475.0,-0.000378,-0.011056
5,3001493,2025,1.188573e+11,13189.5,-0.015785,-0.057741
7,3001495,2025,3.622952e+11,3050.5,0.005046,-0.023215
9,3001496,2025,8.796124e+10,1038.5,-0.323139,-0.047248


In [51]:
FEATURES_CHANGE = [
    "sales_month_avg",
    "sales_yoy",
    "store_avg",
    "store_yoy",
]

X_2025 = df_2025_X[FEATURES_CHANGE].dropna()
X_2025.shape


(1484, 4)

In [52]:
import joblib

model_change = joblib.load("./models/model_change_state.pkl")

y_pred_2025 = model_change.predict(X_2025)
y_proba_2025 = model_change.predict_proba(X_2025)


In [53]:
df_result_2025 = df_2025_X.loc[X_2025.index, ["상권_코드"]].copy()
df_result_2025["pred_change"] = y_pred_2025

# 확률 컬럼도 같이 (선택)
proba_df = pd.DataFrame(
    y_proba_2025,
    columns=model_change.classes_,
    index=X_2025.index
)

df_result_2025 = pd.concat([df_result_2025, proba_df], axis=1)
df_result_2025.head()


,상권_코드,pred_change,다이나믹,상권축소,상권확장,정체
1,3001491,다이나믹,0.506667,0.050000,0.143333,0.300000
3,3001492,정체,0.183333,0.000000,0.006667,0.810000
5,3001493,정체,0.116667,0.000000,0.090000,0.793333
7,3001495,정체,0.220000,0.006667,0.020000,0.753333
9,3001496,정체,0.120000,0.010000,0.066667,0.803333


In [54]:
type(model_change)

sklearn.ensemble._forest.RandomForestClassifier

In [55]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

def make_rf_pipeline():
    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),  # NaN 안전 처리
        ("scaler", StandardScaler()),                   # 있어도 무방(일관성)
        ("model", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ))
    ])


In [56]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib
from pathlib import Path

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

pipe_sales = make_rf_pipeline()
pipe_sales.fit(X_train, y_train)

pred = pipe_sales.predict(X_test)
print(classification_report(y_test, pred))

MODELS_DIR = Path("./models")
MODELS_DIR.mkdir(exist_ok=True)

model_path = MODELS_DIR / "model_sales_direction.pkl"
joblib.dump(pipe_sales, model_path)

print("Saved:", model_path.resolve())


              precision    recall  f1-score   support

         DEC       0.43      0.37      0.40       280
        FLAT       0.27      0.20      0.23       178
         INC       0.54      0.64      0.58       435

    accuracy                           0.47       893
   macro avg       0.41      0.41      0.41       893
weighted avg       0.45      0.47      0.46       893

Saved: C:\Users\user\Documents\GitHub\trade_area_analysis\data\models\model_sales_direction.pkl


In [57]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib
from pathlib import Path

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2,
    test_size=0.2,
    random_state=42,
    stratify=y2
)

pipe_satu = make_rf_pipeline()
pipe_satu.fit(X2_train, y2_train)

pred2 = pipe_satu.predict(X2_test)
print(classification_report(y2_test, pred2))

MODELS_DIR = Path("./models")
MODELS_DIR.mkdir(exist_ok=True)

path2 = MODELS_DIR / "model_saturation.pkl"
joblib.dump(pipe_satu, path2)

print("Saved:", path2.resolve())


              precision    recall  f1-score   support

        HIGH       0.00      0.00      0.00       101
         LOW       0.85      1.00      0.92       760
         MID       0.00      0.00      0.00        32

    accuracy                           0.85       893
   macro avg       0.28      0.33      0.31       893
weighted avg       0.72      0.85      0.78       893

Saved: C:\Users\user\Documents\GitHub\trade_area_analysis\data\models\model_saturation.pkl


In [58]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib
from pathlib import Path

X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y3,
    test_size=0.2,
    random_state=42,
    stratify=y3
)

pipe_change = make_rf_pipeline()
pipe_change.fit(X3_train, y3_train)

pred3 = pipe_change.predict(X3_test)
print(classification_report(y3_test, pred3))

MODELS_DIR = Path("./models")
MODELS_DIR.mkdir(exist_ok=True)

path3 = MODELS_DIR / "model_change_state.pkl"
joblib.dump(pipe_change, path3)

print("Saved:", path3.resolve())


              precision    recall  f1-score   support

        다이나믹       0.48      0.55      0.51       379
        상권축소       0.35      0.29      0.32       266
        상권확장       0.33      0.16      0.21       165
          정체       0.49      0.58      0.53       381

    accuracy                           0.45      1191
   macro avg       0.41      0.40      0.39      1191
weighted avg       0.43      0.45      0.43      1191

Saved: C:\Users\user\Documents\GitHub\trade_area_analysis\data\models\model_change_state.pkl


In [60]:
model_change = joblib.load("./models/model_change_state.pkl")
type(model_change)

sklearn.pipeline.Pipeline